In [3]:
# Importation de la fonction load_dataset depuis Hugging Face Datasets
from datasets import load_dataset

# Chargement d’un dataset local au format CSV (ici un fichier .txt structuré en CSV)
# data_files : chemin vers le fichier de données
# column_names : noms des colonnes du fichier (texte + label)
dataset = load_dataset(
    "csv",
    data_files="../data/sample_texts.csv",
    column_names=["text", "label"]
)

# Affichage du premier exemple du dataset d'entraînement
# dataset["train"] : split principal créé automatiquement par Hugging Face
print(dataset["train"][0])

{'text': 'I absolutely love this product', 'label': 1}


In [4]:
# Fonction de nettoyage des labels
# Convertit les labels en entier (important pour les modèles de classification)
def clean_labels(example):
    example["label"] = int(example["label"])
    return example

# Application de la fonction sur tout le dataset
# map() applique la transformation à chaque ligne du dataset
dataset = dataset.map(clean_labels)

# Affichage du dataset après transformation
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10
    })
})

In [5]:
# Importation du tokenizer automatique de Hugging Face
from transformers import AutoTokenizer

# Définition du modèle pré-entraîné utilisé
model_name = "distilbert-base-uncased"

# Chargement du tokenizer associé au modèle
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fonction de tokenisation des textes du dataset
def tokenize(batch):
    return tokenizer(
        batch["text"],        # colonne contenant les phrases
        padding=True,         # ajoute du padding pour uniformiser les longueurs
        truncation=True       # coupe les textes trop longs
    )

# Application de la tokenisation sur tout le dataset (mode batch pour optimisation)
dataset = dataset.map(tokenize, batched=True)

# Affichage du dataset après tokenisation
dataset

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 10
    })
})

In [6]:
# Renommage de la colonne "label" en "labels"
# Les modèles Hugging Face attendent généralement le nom "labels" pour l'entraînement
dataset = dataset.rename_column("label", "labels")

# Conversion du dataset en format PyTorch
# Cela permet de récupérer directement des tenseurs utilisables par le modèle
dataset.set_format("torch")

In [7]:
# Importation du modèle de classification de séquences pré-entraîné
from transformers import AutoModelForSequenceClassification

# Chargement du modèle DistilBERT adapté à la classification
# num_labels=2 car classification binaire (positif / négatif)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# Importation des classes nécessaires pour l'entraînement
from transformers import TrainingArguments, Trainer

# Définition des paramètres d'entraînement
training_args = TrainingArguments(
    output_dir="./results",              # dossier de sauvegarde des résultats
    num_train_epochs=5,                  # nombre d'époques d'entraînement
    per_device_train_batch_size=8,       # taille du batch par GPU/CPU
    evaluation_strategy="no"             # pas d'évaluation pendant l'entraînement (dataset petit)
)

# Création du Trainer Hugging Face
trainer = Trainer(
    model=model,                         # modèle à entraîner
    args=training_args,                 # paramètres d'entraînement
    train_dataset=dataset["train"]      # dataset d'entraînement
)

c:\Users\adilu\anaconda3\envs\tp4_clean\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [9]:
# Lancement de l'entraînement du modèle
# Le Trainer va optimiser les poids du modèle sur le dataset fourni
trainer.train()

  0%|          | 0/10 [00:00<?, ?it/s]

c:\Users\adilu\anaconda3\envs\tp4_clean\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'train_runtime': 25.1682, 'train_samples_per_second': 1.987, 'train_steps_per_second': 0.397, 'train_loss': 0.6087155342102051, 'epoch': 5.0}


TrainOutput(global_step=10, training_loss=0.6087155342102051, metrics={'train_runtime': 25.1682, 'train_samples_per_second': 1.987, 'train_steps_per_second': 0.397, 'total_flos': 103490155200.0, 'train_loss': 0.6087155342102051, 'epoch': 5.0})

In [10]:
# Texte d'exemple pour tester le modèle entraîné
text = "I love NLP"

# Tokenisation du texte pour le modèle
# return_tensors="pt" : conversion en tenseurs PyTorch
# truncation=True : coupe le texte si trop long
inputs = tokenizer(text, return_tensors="pt", truncation=True)

# Passage du texte dans le modèle pour obtenir les prédictions
outputs = model(**inputs)

# Affichage des logits (scores bruts du modèle avant softmax)
print(outputs.logits)

tensor([[0.1148, 0.0762]], grad_fn=<AddmmBackward0>)


In [11]:
# Importation de PyTorch pour manipuler les tenseurs
import torch

# Liste des classes possibles du modèle
labels = ["negative", "positive"]

# Récupération de la classe prédite (indice avec la plus grande valeur)
pred = torch.argmax(outputs.logits, dim=1)

# Affichage de la prédiction sous forme de label lisible
print("Prediction:", labels[pred])

Prediction: negative
